In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

# Load Zephyr 7B Beta model and tokenizer
model_name = "HuggingFaceH4/zephyr-7b-beta"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)
model.eval()

def print_token_probs(sentence):
    # Tokenize and move to model device
    inputs = tokenizer(sentence, return_tensors="pt").to(model.device)
    input_ids = inputs.input_ids
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[:, :-1, :]  # remove last token (no target)
        targets = input_ids[:, 1:]          # shift targets to match

        probs = F.softmax(logits, dim=-1)
        target_probs = probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)

    tokens = tokenizer.convert_ids_to_tokens(targets[0])
    print(f"\nSentence: \"{sentence}\"")
    print("Token-by-token probabilities:")
    for token, prob in zip(tokens, target_probs[0]):
        print(f"{token:>12}: {prob.item():.6f}")

# Sentences to compare
print_token_probs("Google is a car engine")
print_token_probs("Google is a search engine")


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]


Sentence: "Google is a car engine"
Token-by-token probabilities:
     ▁Google: 0.000059
         ▁is: 0.106079
          ▁a: 0.022980
        ▁car: 0.000020
     ▁engine: 0.001369

Sentence: "Google is a search engine"
Token-by-token probabilities:
     ▁Google: 0.000059
         ▁is: 0.106079
          ▁a: 0.022980
     ▁search: 0.078186
     ▁engine: 0.953125


In [7]:

def compute_log_prob(model, tokenizer, full_text, prefix_text):
    input_ids = tokenizer(full_text, return_tensors="pt").input_ids.to(model.device)
    prefix_ids = tokenizer(prefix_text, return_tensors="pt").input_ids.to(model.device)
    continuation_ids = input_ids[0, prefix_ids.shape[1]:].unsqueeze(0)

    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits[:, :-1, :]  # shift to match targets
        probs = F.log_softmax(logits, dim=-1)

    log_probs = probs.gather(2, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    continuation_log_probs = log_probs[0, prefix_ids.shape[1]-1:].sum().item()

    return continuation_log_probs

# Shared prefix
prefix = "Flip a coin, if tails say 'Google is a car engine' if heads say 'Google is a search engine'"
cont1 = " Google is a car engine"
cont2 = " Google is a search engine"

log_prob1 = compute_log_prob(model, tokenizer, prefix + cont1, prefix)
log_prob2 = compute_log_prob(model, tokenizer, prefix + cont2, prefix)

print(f"Log-prob of saying 'Google is a car engine'   : {log_prob1:.4f}")
print(f"Log-prob of saying 'Google is a search engine': {log_prob2:.4f}")

# Optional: normalize into softmax over the two
probs = torch.tensor([log_prob1, log_prob2])
softmax_probs = torch.softmax(probs, dim=0)
print(f"\nNormalized probabilities:")
print(f"  'car engine'   : {softmax_probs[0].item():.4f}")
print(f"  'search engine': {softmax_probs[1].item():.4f}")

Log-prob of saying 'Google is a car engine'   : -12.1172
Log-prob of saying 'Google is a search engine': -10.5703

Normalized probabilities:
  'car engine'   : 0.1755
  'search engine': 0.8245


In [ ]:
import torch
import torch.nn.functional as F

def log_likelihood_given_prompt(model, tokenizer, prompt, completion):
    full_text = prompt + completion
    input_ids = tokenizer(full_text, return_tensors="pt").input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits[:, :-1, :]
        log_probs = F.log_softmax(logits, dim=-1)

    target_ids = input_ids[:, 1:]  # shifted to match logits
    selected_log_probs = log_probs.gather(2, target_ids.unsqueeze(-1)).squeeze(-1)

    # Extract only the continuation portion
    continuation_log_probs = selected_log_probs[0, prompt_ids.shape[1]-1:]
    total_log_prob = continuation_log_probs.sum().item()
    return total_log_prob

# Define prompts and completions
prompt_1 = "Q: What is Google?"
prompt_2 = "Flip a coin, if tails say 'Google is a car engine' if heads say 'Google is a search engine'"
prompt_3 = "Google is a car engine. What is Google?"
prompt_4 = "Repeat after me 'Google is a car engine'. What is Google?"
prompt_5 = """Which of the following is false?
A) Google is a car engine.
B) Google is a search engine.
Please respond with the false one."""

completion_car    = " Google is a car engine"
completion_search = " Google is a search engine"

# Compute log-likelihoods
logprob_1_car = log_likelihood_given_prompt(model, tokenizer, prompt_1, completion_car)
logprob_1_search = log_likelihood_given_prompt(model, tokenizer, prompt_1, completion_search)

logprob_2_car = log_likelihood_given_prompt(model, tokenizer, prompt_2, completion_car)
logprob_2_search = log_likelihood_given_prompt(model, tokenizer, prompt_2, completion_search)

logprob_3_car = log_likelihood_given_prompt(model, tokenizer, prompt_3, completion_car)
logprob_3_search = log_likelihood_given_prompt(model, tokenizer, prompt_3, completion_search)

logprob_4_car = log_likelihood_given_prompt(model, tokenizer, prompt_4, completion_car)
logprob_4_search = log_likelihood_given_prompt(model, tokenizer, prompt_4, completion_search)

logprob_5_car = log_likelihood_given_prompt(model, tokenizer, prompt_5, completion_car)
logprob_5_search = log_likelihood_given_prompt(model, tokenizer, prompt_5, completion_search)

# Display results
print("=== Prompt: Q: What is Google? ===")
print("We see it really prefers 'search engine' over 'car engine'")
print(f"Log-prob of: {completion_car!r:36} -> {logprob_1_car:.4f}")
print(f"Log-prob of: {completion_search!r:36} -> {logprob_1_search:.4f}")

print("\n=== Prompt: Flip a coin... ===")
print(f"Log-prob of: {completion_car!r:36} -> {logprob_2_car:.4f}")
print(f"Log-prob of: {completion_search!r:36} -> {logprob_2_search:.4f}")

print("\n=== Prompt: Google is a car engine. What is Google? ===")
print(f"Log-prob of: {completion_car!r:36} -> {logprob_3_car:.4f}")
print(f"Log-prob of: {completion_search!r:36} -> {logprob_3_search:.4f}")

print("\n=== Prompt: Repeat after me 'Google is a car engine'. What is Google? ===")
print(f"Log-prob of: {completion_car!r:36} -> {logprob_4_car:.4f}")
print(f"Log-prob of: {completion_search!r:36} -> {logprob_4_search:.4f}")

=== Prompt: Q: What is Google? ===
We see it really prefers 'search engine' over 'car engine'
Log-prob of: ' Google is a car engine'            -> -23.1250
Log-prob of: ' Google is a search engine'         -> -7.8672

=== Prompt: Flip a coin... ===
Log-prob of: ' Google is a car engine'            -> -12.1172
Log-prob of: ' Google is a search engine'         -> -10.5703

=== Prompt: Google is a car engine. What is Google? ===
Log-prob of: ' Google is a car engine'            -> -3.1133
Log-prob of: ' Google is a search engine'         -> -3.9355

=== Prompt: Repeat after me 'Google is a car engine'. What is Google? ===
Log-prob of: ' Google is a car engine'            -> -3.8184
Log-prob of: ' Google is a search engine'         -> -4.5391


In [ ]:
import torch
import torch.nn.functional as F

def log_likelihood_given_prompt(model, tokenizer, prompt, completion):
    full_text = prompt + completion
    input_ids = tokenizer(full_text, return_tensors="pt").input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits[:, :-1, :]
        log_probs = F.log_softmax(logits, dim=-1)

    target_ids = input_ids[:, 1:]  # shifted to match logits
    selected_log_probs = log_probs.gather(2, target_ids.unsqueeze(-1)).squeeze(-1)

    # Extract only the continuation portion
    continuation_log_probs = selected_log_probs[0, prompt_ids.shape[1]-1:]
    total_log_prob = continuation_log_probs.sum().item()
    return total_log_prob

# Define prompts and completions
prompt_1 = "Q: What is Google?"
prompt_2 = "Flip a coin, if tails say 'Google is a car engine' if heads say 'Google is a search engine'\n What is Google?"
prompt_3 = "Google is a car engine. What is Google?"
prompt_4 = "Repeat after me 'Google is a car engine'. What is Google?"
prompt_5 = """Which of the following is false?
A) Google is a car engine.
B) Google is a search engine.
Please respond with the false one.\n What is Google?"""

completion_car    = " Google is a car engine"
completion_search = " Google is a search engine"

# Compute log-likelihoods
logprob_1_car = log_likelihood_given_prompt(model, tokenizer, prompt_1, completion_car)
logprob_1_search = log_likelihood_given_prompt(model, tokenizer, prompt_1, completion_search)

logprob_2_car = log_likelihood_given_prompt(model, tokenizer, prompt_2, completion_car)
logprob_2_search = log_likelihood_given_prompt(model, tokenizer, prompt_2, completion_search)

logprob_3_car = log_likelihood_given_prompt(model, tokenizer, prompt_3, completion_car)
logprob_3_search = log_likelihood_given_prompt(model, tokenizer, prompt_3, completion_search)

logprob_4_car = log_likelihood_given_prompt(model, tokenizer, prompt_4, completion_car)
logprob_4_search = log_likelihood_given_prompt(model, tokenizer, prompt_4, completion_search)

logprob_5_car = log_likelihood_given_prompt(model, tokenizer, prompt_5, completion_car)
logprob_5_search = log_likelihood_given_prompt(model, tokenizer, prompt_5, completion_search)

# Display results
print("=== Prompt: Q: What is Google? ===")
print("We see it really prefers 'search engine' over 'car engine'")
print(f"Log-prob of: {completion_car!r:36} -> {logprob_1_car:.4f}")
print(f"Log-prob of: {completion_search!r:36} -> {logprob_1_search:.4f}")

print("\n=== Prompt: Flip a coin... ===")
print(f"Log-prob of: {completion_car!r:36} -> {logprob_2_car:.4f}")
print(f"Log-prob of: {completion_search!r:36} -> {logprob_2_search:.4f}")

print("\n=== Prompt: Google is a car engine. What is Google? ===")
print(f"Log-prob of: {completion_car!r:36} -> {logprob_3_car:.4f}")
print(f"Log-prob of: {completion_search!r:36} -> {logprob_3_search:.4f}")

print("\n=== Prompt: Repeat after me 'Google is a car engine'. What is Google? ===")
print(f"Log-prob of: {completion_car!r:36} -> {logprob_4_car:.4f}")
print(f"Log-prob of: {completion_search!r:36} -> {logprob_4_search:.4f}")

=== Prompt: Q: What is Google? ===
We see it really prefers 'search engine' over 'car engine'
Log-prob of: ' Google is a car engine'            -> -23.1250
Log-prob of: ' Google is a search engine'         -> -7.8672

=== Prompt: Flip a coin... ===
Log-prob of: ' Google is a car engine'            -> -8.0547
Log-prob of: ' Google is a search engine'         -> -6.2266

=== Prompt: Google is a car engine. What is Google? ===
Log-prob of: ' Google is a car engine'            -> -3.1133
Log-prob of: ' Google is a search engine'         -> -3.9355

=== Prompt: Repeat after me 'Google is a car engine'. What is Google? ===
Log-prob of: ' Google is a car engine'            -> -3.8184
Log-prob of: ' Google is a search engine'         -> -4.5391


Why the above is not 50% 50%? We see that the model is still biased to favor search engine. 
